# Overfitting in Neural Networks and CNNs — Complete Classroom Lab

**From memorization to generalization | PyTorch | Google Colab / Jupyter | No dataset download**

### What we will demonstrate
1. **Overfitting in a fully connected neural network (MLP)** — watch training loss fall while validation loss rises.
2. **Overfitting in a CNN** — even convolutional weight sharing does not prevent memorization.
3. Test **dropout**, **Dropout2d**, **weight decay**, **early stopping**, **smaller models**, **data augmentation**, and **more training data**, and **cleaner labels**.
4. **Transfer learning** — reuse a CNN trained on one digit task for a related task with little labeled data.
5. Compare experiments fairly and evaluate a selected model on a **held-out test set**.

> **Why this dataset?** We use `sklearn.datasets.load_digits()`: 1,797 small handwritten digit images (8×8), already packaged with scikit-learn. **No download, no 403 error**. This is different from the 28×28 MNIST dataset, but uses the same digit-classification concepts.

**Classroom time:** ~60–90 minutes including explanations and experiments. **Compute:** CPU is sufficient.  
**Important:** For a clear demonstration, we intentionally train on a **small sample with 25% deliberately incorrect training labels**. Validation/test labels stay clean. The resulting overfitting is educational and deliberately exaggerated.

## 0. Setup

Google Colab normally includes these libraries. If your environment doesn't, uncomment the installation line. No image datasets are fetched from the internet.

In [ ]:
# Uncomment only if any packages are missing:
# %pip install -q torch scikit-learn matplotlib pandas numpy

import copy
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(2, torch.get_num_threads()))
DEVICE = torch.device("cpu")  # Very small dataset; CPU keeps results easy to reproduce.
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 1. Create a realistic train / validation / test split

**On your writing pad:**
- **Training**: learns the weights.
- **Validation**: helps choose settings/checkpoints; not used to update weights.
- **Test**: untouched until the final evaluation.
- **Overfitting**: training-specific patterns are learned at the expense of generalization.

We reserve **20% for validation and 20% for testing**, stratifying by digit so class proportions remain similar.

In [ ]:
digits = load_digits()
images_np = digits.images.astype("float32") / 16.0  # Pixel range becomes 0..1
labels_clean = digits.target.astype("int64")
images = torch.tensor(images_np[:, None, :, :])  # (N, 1, 8, 8)

all_ids = np.arange(len(labels_clean))
train_val_ids, test_ids = train_test_split(
    all_ids, test_size=0.20, stratify=labels_clean, random_state=SEED
)
train_pool_ids, val_ids = train_test_split(
    train_val_ids, test_size=0.25,
    stratify=labels_clean[train_val_ids], random_state=SEED
)

rng = np.random.default_rng(SEED)
train_pool_ids = rng.permutation(train_pool_ids)
small_train_ids = train_pool_ids[:120]
large_train_ids = train_pool_ids[:800]

# Deliberately corrupt 25% of labels in the TRAINING pool only.
labels_noisy = labels_clean.copy()
corrupted_ids = rng.choice(train_pool_ids, size=len(train_pool_ids)//4, replace=False)
labels_noisy[corrupted_ids] = (
    labels_clean[corrupted_ids] + rng.integers(1, 10, size=len(corrupted_ids))
) % 10

print("All images:", len(images))
print("Small training set:", len(small_train_ids))
print("Larger training set:", len(large_train_ids))
print("Validation set:", len(val_ids), "| Test set:", len(test_ids))
print("Wrong training labels in small set:",
      f"{np.mean(labels_noisy[small_train_ids] != labels_clean[small_train_ids]):.0%}")
print("Validation and test labels: untouched")
assert not set(train_pool_ids).intersection(set(val_ids))
assert not set(train_pool_ids).intersection(set(test_ids))
assert not set(val_ids).intersection(set(test_ids))

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(13, 4))
for ax, idx in zip(axes.flat, small_train_ids[:14]):
    ax.imshow(images_np[idx], cmap="gray")
    true_label = labels_clean[idx]
    seen_label = labels_noisy[idx]
    label_text = f"True: {true_label} | Train: {seen_label}"
    ax.set_title(label_text, fontsize=9)
    ax.axis("off")
plt.suptitle("Some training labels are deliberately wrong (look for mismatches)")
plt.tight_layout()
plt.show()

**Ask students:** If a model reaches 100% accuracy against these *noisy training labels*, is it necessarily learning genuine digit patterns? **No.** It may be memorizing incorrect examples too.

## 2. Define two models: MLP and CNN

- **MLP:** flatten 8×8 to 64 inputs, then fully connected layers.
- **CNN:** local convolution filters, ReLU, pooling, and a classifier.
- Both can use dropout. For the CNN, **`Dropout2d` randomly drops entire feature-map channels** during training.

**Key difference:** `nn.Dropout` acts on individual activations; `nn.Dropout2d` acts on whole channels for 4D convolution outputs. Neither permanently deletes filters.

In [ ]:
class MLP(nn.Module):
    def __init__(self, small=False, dropout=False, num_classes=10):
        super().__init__()
        h1, h2 = (32, 16) if small else (256, 128)
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, h1), nn.ReLU(),
            nn.Dropout(0.4) if dropout else nn.Identity(),
            nn.Linear(h1, h2), nn.ReLU(),
            nn.Dropout(0.4) if dropout else nn.Identity(),
            nn.Linear(h2, num_classes)
        )

    def forward(self, x):
        return self.network(x)


class SimpleCNN(nn.Module):
    def __init__(self, small=False, dropout=False, num_classes=10):
        super().__init__()
        c1, c2 = (8, 16) if small else (32, 64)
        hidden = 32 if small else 128
        self.features = nn.Sequential(
            nn.Conv2d(1, c1, 3, padding=1), nn.ReLU(),
            nn.Conv2d(c1, c2, 3, padding=1), nn.ReLU(),
            nn.Dropout2d(0.2) if dropout else nn.Identity(),
            nn.MaxPool2d(2),
            nn.Conv2d(c2, c2, 3, padding=1), nn.ReLU(),
            nn.Dropout2d(0.2) if dropout else nn.Identity(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(c2 * 2 * 2, hidden), nn.ReLU(),
            nn.Dropout(0.4) if dropout else nn.Identity(),
        )
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("MLP trainable parameters:", count_params(MLP()))
print("CNN trainable parameters:", count_params(SimpleCNN()))
print("CNN output for one image:", tuple(SimpleCNN()(images[:1]).shape))

## 3. One transparent training function for all experiments

**Four steps in every batch:** forward → loss → `backward()` → optimizer step.

The function also records **train and validation accuracy/loss after each epoch**. The training evaluation uses the *same noisy labels the model was trained on*, so it reveals memorization. Validation uses correct unseen labels.

We use **AdamW** so that the weight-decay experiment demonstrates decoupled weight decay.

In [ ]:
def augment_batch(x):
    """Small random rotations and translations, only for training images."""
    n = x.shape[0]
    angle = (torch.rand(n) - 0.5) * 0.30     # about +/- 8.6 degrees
    tx = (torch.rand(n) - 0.5) * 0.14        # small horizontal shift
    ty = (torch.rand(n) - 0.5) * 0.14        # small vertical shift
    theta = torch.zeros(n, 2, 3)
    theta[:, 0, 0] = torch.cos(angle)
    theta[:, 0, 1] = -torch.sin(angle)
    theta[:, 1, 0] = torch.sin(angle)
    theta[:, 1, 1] = torch.cos(angle)
    theta[:, 0, 2] = tx
    theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, align_corners=False, padding_mode="zeros")


def evaluate(model, ids, labels):
    model.eval()   # IMPORTANT: disables dropout for evaluation
    with torch.inference_mode():
        y = torch.tensor(labels[ids], dtype=torch.long)
        logits = model(images[ids])
        loss = F.cross_entropy(logits, y).item()
        accuracy = (logits.argmax(1) == y).float().mean().item()
    return loss, accuracy


def fit(model, train_ids, train_labels, val_labels=labels_clean,
        epochs=100, lr=0.003, weight_decay=0.0,
        augment=False, patience=None):
    torch.manual_seed(123)
    model = model.to(DEVICE)
    ds = TensorDataset(
        images[train_ids], torch.tensor(train_labels[train_ids], dtype=torch.long)
    )
    loader = DataLoader(ds, batch_size=32, shuffle=True)
    optimizer = torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=lr, weight_decay=weight_decay
    )
    history = []
    best_loss = float("inf")
    best_epoch = 0
    best_state = None
    stale_epochs = 0
    for epoch in range(1, epochs + 1):
        model.train()  # IMPORTANT: activates dropout during training
        for x, y in loader:
            if augment:
                x = augment_batch(x)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()

        train_loss, train_acc = evaluate(model, train_ids, train_labels)
        val_loss, val_acc = evaluate(model, val_ids, val_labels)
        history.append(dict(
            epoch=epoch, train_loss=train_loss, val_loss=val_loss,
            train_acc=train_acc, val_acc=val_acc
        ))
        if val_loss < best_loss - 0.001:
            best_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
        if patience is not None and stale_epochs >= patience:
            break

    # Restore best checkpoint ONLY in the early-stopping experiment.
    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)
    return {
        "model": model, "history": pd.DataFrame(history),
        "best_epoch": best_epoch, "best_val_loss": best_loss,
        "params": count_params(model)
    }


def plot_history(result, title):
    df = result["history"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(df.epoch, df.train_loss, label="Train")
    axes[0].plot(df.epoch, df.val_loss, label="Validation")
    axes[0].set(title=f"{title}: Loss", xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[0].legend()
    axes[1].plot(df.epoch, df.train_acc * 100, label="Train")
    axes[1].plot(df.epoch, df.val_acc * 100, label="Validation")
    axes[1].set(title=f"{title}: Accuracy", xlabel="Epoch", ylabel="Accuracy (%)", ylim=(0, 105))
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 4. Demonstration A — Overfitting in a fully connected NN (MLP)

**Prediction:** Training loss will become very small and training accuracy can approach 100%, yet validation loss may **rise**. This is the exact behavior we want students to recognize.

In [ ]:
experiments = {}
start = time.time()
experiments["MLP / Baseline"] = fit(MLP(), small_train_ids, labels_noisy, epochs=90)
plot_history(experiments["MLP / Baseline"], "MLP baseline")
print("Time:", round(time.time() - start, 1), "seconds")
print("Last epoch:", experiments["MLP / Baseline"]["history"].tail(1).round(3).to_string(index=False))

**Ask students:** If the model reaches 100% training accuracy *against intentionally wrong labels*, is more training the solution? **No**—we may be fitting label noise and the validation curve tells us when generalization deteriorates.

## 5. Demonstration B — Overfitting in a CNN

CNNs share filter weights and have a spatial bias, **but they can still overfit** small, noisy datasets.

In [ ]:
start = time.time()
experiments["CNN / Baseline"] = fit(SimpleCNN(), small_train_ids, labels_noisy, epochs=90)
plot_history(experiments["CNN / Baseline"], "CNN baseline")
print("Time:", round(time.time() - start, 1), "seconds")
print("Last epoch:", experiments["CNN / Baseline"]["history"].tail(1).round(3).to_string(index=False))

## 6. Show what `Dropout2d` actually does

**Standard dropout:** randomly zeroes individual activations.  
**Spatial dropout (`Dropout2d`):** randomly zeroes **entire feature-map channels** in a CNN.  
**Not the same as deleting or changing the filter weights.**

PyTorch scales surviving activations during training; the illustration prints whether each channel survives.

In [ ]:
torch.manual_seed(7)
drop = nn.Dropout2d(p=0.5)
drop.train()
example_maps = torch.ones(1, 8, 4, 4)  # 8 feature maps of size 4x4
masked = drop(example_maps)
print("Channel mean after Dropout2d:")
print(masked.mean(dim=(2, 3)).numpy().round(1))
print("Zero means an entire channel was dropped; kept channels may be scaled up.")
drop.eval()
print("In evaluation mode, the output is unchanged:",
      torch.equal(drop(example_maps), example_maps))

## 7. Run **one fix at a time** (MLP and CNN)

We change one setting at a time to make the effects easier to discuss. The methods are **not guaranteed to improve every task**. On tiny 8×8 digits, augmentation can even remove helpful detail.

| Fix | How to explain it |
|---|---|
| Dropout | Temporarily hide activations during training |
| Dropout2d | Temporarily hide whole CNN channels |
| Weight decay | Discourage overly large parameter values |
| Early stopping | Stop when validation loss stops improving, restore best checkpoint |
| Smaller model | Reduce number of trainable parameters |
| Data augmentation | Train with small altered versions of images |
| More data | Show more distinct examples; still use same label-noise process |
| Correct labels | Repair mislabeled training examples |

**Note:** The “more data” run uses more batches per epoch and therefore more optimization steps. The experiment illustrates a practical intervention, not a perfectly matched-compute comparison.

In [ ]:
# MLP fixes — each run uses the same small set unless stated otherwise.
mlp_cases = [
    ("MLP / Dropout", dict(dropout=True)),
    ("MLP / Weight decay", dict(weight_decay=0.1)),
    ("MLP / Early stopping", dict(patience=10)),
    ("MLP / Smaller model", dict(small=True)),
    ("MLP / Augmentation", dict(augment=True)),
    ("MLP / More data", dict(more_data=True, epochs=20)),
    ("MLP / Correct labels", dict(clean_labels=True)),
]
for title, cfg in mlp_cases:
    cfg = cfg.copy()
    model = MLP(small=cfg.pop("small", False), dropout=cfg.pop("dropout", False))
    train_ids = large_train_ids if cfg.pop("more_data", False) else small_train_ids
    train_targets = labels_clean if cfg.pop("clean_labels", False) else labels_noisy
    experiments[title] = fit(model, train_ids, train_targets,
                             epochs=cfg.pop("epochs", 60), **cfg)
    result = experiments[title]
    df = result["history"]
    print(f"{title:<23} | epochs={len(df):3d} | "
          f"train={df.train_acc.iloc[-1]:.0%} | val={df.val_acc.iloc[-1]:.0%} | "
          f"best epoch={result['best_epoch']}")

In [ ]:
# CNN fixes — Dropout2d is used inside the CNN's feature extractor.
cnn_cases = [
    ("CNN / Spatial dropout", dict(dropout=True)),
    ("CNN / Weight decay", dict(weight_decay=0.1)),
    ("CNN / Early stopping", dict(patience=10)),
    ("CNN / Smaller model", dict(small=True)),
    ("CNN / Augmentation", dict(augment=True)),
    ("CNN / More data", dict(more_data=True, epochs=20)),
    ("CNN / Correct labels", dict(clean_labels=True)),
    ("CNN / Combined fixes", dict(dropout=True, weight_decay=0.05,
                                   augment=True, patience=10)),
]
for title, cfg in cnn_cases:
    cfg = cfg.copy()
    model = SimpleCNN(small=cfg.pop("small", False),
                      dropout=cfg.pop("dropout", False))
    train_ids = large_train_ids if cfg.pop("more_data", False) else small_train_ids
    train_targets = labels_clean if cfg.pop("clean_labels", False) else labels_noisy
    experiments[title] = fit(model, train_ids, train_targets,
                             epochs=cfg.pop("epochs", 60), **cfg)
    result = experiments[title]
    df = result["history"]
    print(f"{title:<23} | epochs={len(df):3d} | "
          f"train={df.train_acc.iloc[-1]:.0%} | val={df.val_acc.iloc[-1]:.0%} | "
          f"best epoch={result['best_epoch']}")

## 8. Side-by-side comparison of fixes

Interpret **validation loss**, not just training accuracy. The early-stopping runs load the **best validation checkpoint**, while other rows describe their final epoch. This is deliberate for teaching; it is **not** a rigorous scientific benchmark. A fair deployment comparison would select checkpoints and tune each method consistently.

In [ ]:
rows = []
for name, result in experiments.items():
    h = result["history"]
    train_acc = h.train_acc.iloc[-1]
    # For early stopping, the model has been restored to the best checkpoint.
    if "Early stopping" in name or "Combined fixes" in name:
        val_loss, val_acc = evaluate(result["model"], val_ids, labels_clean)
        train_ids = small_train_ids
        train_labels = labels_noisy
        _, train_acc = evaluate(result["model"], train_ids, train_labels)
    else:
        val_loss, val_acc = h.val_loss.iloc[-1], h.val_acc.iloc[-1]
    rows.append({
        "Experiment": name,
        "Epochs": len(h),
        "Parameters": result["params"],
        "Training accuracy %": round(train_acc * 100, 1),
        "Validation accuracy %": round(val_acc * 100, 1),
        "Validation loss": round(val_loss, 3),
        "Best val-loss epoch": result["best_epoch"]
    })
summary = pd.DataFrame(rows)
display(summary.sort_values(["Experiment"]).reset_index(drop=True))

In [ ]:
# Plot the validation-loss curves for the baseline and selected fixes.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, prefix, choices in [
    (axes[0], "MLP", ["Baseline", "Dropout", "Weight decay", "Early stopping", "More data"]),
    (axes[1], "CNN", ["Baseline", "Spatial dropout", "Weight decay", "Early stopping", "More data", "Combined fixes"]),
]:
    for choice in choices:
        h = experiments[f"{prefix} / {choice}"]["history"]
        ax.plot(h.epoch, h.val_loss, label=choice)
    ax.set(title=f"{prefix}: Validation loss", xlabel="Epoch", ylabel="Loss")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Discuss before continuing

1. **Did training loss decrease?** Does that alone prove a model is good?
2. Which methods improved **validation** results in *this* run? What happened when we corrected the noisy labels?
3. Why might **augmentation** hurt on an 8×8 digit if the rotation/translation is too strong?
4. Does **dropout** delete the CNN's filters? **No**. It masks activations during training.
5. Is the “more data” result solely caused by more images? **Not entirely**; it also changes the number of updates and exposes more varied examples.

**Key learning:** Regularization is a toolbox, **not a guarantee**. The validation set helps us decide what works.

## 9. Transfer learning — reuse visual knowledge on a new task

**Goal:** First pretrain a CNN to classify **digits 0–9**, then reuse its **frozen feature extractor** for a related task: **Even vs Odd** (two classes), with very few new labeled examples.

We use **different training images** for pretraining and the new task. The validation and test sets are not used for fitting either model.

This is a *small teaching demonstration of transfer learning*, not a pretrained ImageNet model such as ResNet18. For a production image project, pretrained weights generally come from a larger external dataset.

In [ ]:
# Disjoint training images for source (10-digit) and target (even/odd) tasks.
source_ids = train_pool_ids[:800]
target_ids = train_pool_ids[800:880]  # Only 80 labeled examples on the new task
labels_even_odd = (labels_clean % 2).astype("int64")  # 0=even, 1=odd
assert not set(source_ids).intersection(set(target_ids))

print("Pretraining examples:", len(source_ids))
print("Transfer-task training examples:", len(target_ids))

pretrained = fit(
    SimpleCNN(num_classes=10), source_ids, labels_clean,
    epochs=25, lr=0.003
)
print("Pretraining done. Last source-task validation accuracy:",
      f"{pretrained['history'].val_acc.iloc[-1]:.1%}")

# Transfer: same feature weights, NEW 2-class head.
transfer_model = SimpleCNN(num_classes=2)
transfer_model.features.load_state_dict(pretrained["model"].features.state_dict())
for p in transfer_model.features.parameters():
    p.requires_grad = False

# Baseline on same 80 target images: learn from scratch.
scratch_model = SimpleCNN(num_classes=2)
print("Trainable parameters (scratch):", count_params(scratch_model))
print("Trainable parameters (frozen-backbone transfer):", count_params(transfer_model))

In [ ]:
scratch_result = fit(
    scratch_model, target_ids, labels_even_odd,
    val_labels=labels_even_odd, epochs=40, lr=0.003,
    patience=10
)
transfer_result = fit(
    transfer_model, target_ids, labels_even_odd,
    val_labels=labels_even_odd, epochs=40, lr=0.005,
    patience=10
)

comparison = []
for name, r in [("Train from scratch", scratch_result),
                ("Frozen pretrained CNN", transfer_result)]:
    val_loss, val_acc = evaluate(r["model"], val_ids, labels_even_odd)
    comparison.append({"Model": name,
                       "Trainable parameters": r["params"],
                       "Validation accuracy %": round(val_acc * 100, 1),
                       "Validation loss": round(val_loss, 3),
                       "Epochs": len(r["history"])})
display(pd.DataFrame(comparison))

fig, ax = plt.subplots(figsize=(8, 3.5))
for name, r in [("Scratch", scratch_result), ("Transfer (frozen)", transfer_result)]:
    ax.plot(r["history"].epoch, r["history"].val_acc * 100, label=name)
ax.set(title="Transfer learning: Even vs Odd validation accuracy",
       xlabel="Epoch", ylabel="Accuracy (%)", ylim=(0, 105))
ax.legend()
plt.show()

**Teaching notes:**
- Pretraining learns a useful representation from a related task.
- Freezing means pretrained parameters **do not update** during the new task.
- The new classification head **does update**.
- Transfer may or may not beat scratch training; **task similarity matters**. If you choose to unfreeze later layers and continue training, that is **fine-tuning**.
- Do not confuse this notebook's simple transfer demo with fine-tuning a large pretrained LLM; the broad principle of reusing learned representations is similar.

## 10. One honest final test (no peeking during model selection)

For an illustrative final evaluation, select the CNN experiment with the **lowest validation loss at the final/retained checkpoint**, and evaluate it **once** on the held-out test set.

**We never train on test data.** If we change the model after inspecting test results repeatedly, the test set is no longer an unbiased final evaluation.

In [ ]:
cnn_names = [name for name in experiments if name.startswith("CNN / ")]
cnn_val_losses = {
    name: evaluate(experiments[name]["model"], val_ids, labels_clean)[0]
    for name in cnn_names
}
chosen_name = min(cnn_val_losses, key=cnn_val_losses.get)
chosen_model = experiments[chosen_name]["model"]
validation_loss, validation_acc = evaluate(chosen_model, val_ids, labels_clean)
test_loss, test_acc = evaluate(chosen_model, test_ids, labels_clean)
print("Selected by validation loss:", chosen_name)
print(f"Validation accuracy: {validation_acc:.1%} | Test accuracy: {test_acc:.1%}")
print(f"Validation loss: {validation_loss:.3f} | Test loss: {test_loss:.3f}")

chosen_model.eval()
with torch.inference_mode():
    predictions = chosen_model(images[test_ids]).argmax(dim=1).numpy()
cm = confusion_matrix(labels_clean[test_ids], predictions, labels=list(range(10)))
fig, ax = plt.subplots(figsize=(6.5, 5))
ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(ax=ax, colorbar=False)
ax.set_title("Held-out test confusion matrix")
plt.show()

## 11. Industry checklist and common mistakes

**Before deploying any NN or CNN:**

- Split the data correctly (keep related photos/patients/customers in the same split).
- Fit preprocessing statistics **only on the training split**, if statistics are learned.
- Inspect **training and validation** learning curves together.
- Consider more representative labeled data and correcting bad labels.
- Use label-preserving augmentation **only during training**.
- Test dropout, weight decay, early stopping and a simpler architecture.
- Try a pretrained backbone when data is scarce and the tasks are related.
- Select using validation metrics; use the untouched test set for final assessment.
- Monitor model quality after deployment: real-world data can change.

**A low training loss does not guarantee real-world performance.**

### Discussion exercises
1. Change the number of small training examples from `120` to `60`. What happens to overfitting?
2. Remove label corruption (use `labels_clean` as training labels). Do the gaps shrink?
3. Try `nn.Dropout(0.2)` versus `0.6` in MLP. What happens to underfitting?
4. Change CNN `Dropout2d(0.2)` to `0.5`. Is stronger always better?
5. Try `weight_decay=0.01` versus `0.1` and compare validation loss.
6. Make augmentation too strong. Why can it harm 8×8 handwritten digits?
7. Explain in one sentence why **feature extraction** differs from **fine-tuning**.

### Final board takeaway
**Overfitting = excellent fit to training-specific details, weaker generalization.**  
**Solutions = better data + sensible capacity + regularization + correct evaluation.**

*This is a demonstration notebook. Results may vary with library versions, random seeds and hardware. Our small dataset and corrupted labels deliberately exaggerate memorization so students can see it clearly.*